In [1]:
import os
from typing import Literal, Optional

import arxiv
import requests
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv()

for key in ["GROQ_API_KEY", "OPENALEX_API_KEY"]:
    assert os.environ.get(key), f"Missing {key} in .env"

print("Environment loaded ✅")

Environment loaded ✅


In [2]:
class Paper(BaseModel):
    title: str
    authors: list[str]
    year: int | None = None
    abstract: str | None = None
    url: str | None = None
    pdf_url: str | None = None
    citation_count: int | None = None
    source: Literal["arxiv", "openalex", "semantic_scholar"]

In [3]:
import time

_last_ss_call = 0.0
SEMANTIC_SCHOLAR_BASE = "https://api.semanticscholar.org/graph/v1/paper/search"

def search_semantic_scholar(query: str, max_results: int = 5) -> list[Paper]:
    global _last_ss_call
    elapsed = time.time() - _last_ss_call
    if elapsed < 1.0:
        time.sleep(1.0 - elapsed)
    _last_ss_call = time.time()

    headers = {"x-api-key": os.environ["SEMANTIC_SCHOLAR_API_KEY"]}
    params = {
        "query": query,
        "limit": max_results,
        "fields": "title,year,abstract,authors,citationCount,openAccessPdf",
    }
    try:
        resp = requests.get(SEMANTIC_SCHOLAR_BASE, params=params, headers=headers, timeout=15)
        resp.raise_for_status()
        data = resp.json().get("data", [])
    except Exception as e:
        print(f"⚠️ Semantic Scholar search failed ({type(e).__name__}); skipping")
        return []

    return [
        Paper(
            title=p.get("title") or "Untitled",
            authors=[a["name"] for a in p.get("authors", [])],
            year=p.get("year"),
            abstract=p.get("abstract"),
            url=f"https://www.semanticscholar.org/paper/{p['paperId']}" if p.get("paperId") else None,
            pdf_url=(p.get("openAccessPdf") or {}).get("url"),
            citation_count=p.get("citationCount"),
            source="semantic_scholar",
        )
        for p in data
    ]

In [4]:
def search_arxiv(query: str, max_results: int = 5) -> list[Paper]:
    client = arxiv.Client(page_size=max_results, delay_seconds=3.0, num_retries=2)
    search = arxiv.Search(
        query=query, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance
    )
    try:
        return [
            Paper(
                title=r.title,
                authors=[a.name for a in r.authors],
                year=r.published.year,
                abstract=r.summary.replace("\n", " "),
                url=r.entry_id,
                pdf_url=r.pdf_url,
                source="arxiv",
            )
            for r in client.results(search)
        ]
    except Exception as e:
        print(
            f"⚠️ arXiv search failed ({type(e).__name__}); continuing with OpenAlex only"
        )
        return []

In [5]:
OPENALEX_BASE = "https://api.openalex.org/works"


def _reconstruct_abstract(inverted_index: dict | None) -> str | None:
    if not inverted_index:
        return None
    positions = {}
    for word, idxs in inverted_index.items():
        for idx in idxs:
            positions[idx] = word
    return " ".join(positions[i] for i in sorted(positions))


def search_openalex(query: str, max_results: int = 5) -> list[Paper]:
    params = {
        "search": query,
        "per_page": max_results,
        "select": "title,authorships,publication_year,abstract_inverted_index,id,open_access,cited_by_count",
        "api_key": os.environ["OPENALEX_API_KEY"],
    }
    resp = requests.get(OPENALEX_BASE, params=params, timeout=15)
    resp.raise_for_status()

    results = []
    for w in resp.json()["results"]:
        results.append(
            Paper(
                title=w.get("title") or "Untitled",
                authors=[a["author"]["display_name"] for a in w.get("authorships", [])],
                year=w.get("publication_year"),
                abstract=_reconstruct_abstract(w.get("abstract_inverted_index")),
                url=w.get("id"),
                pdf_url=(w.get("open_access") or {}).get("oa_url"),
                citation_count=w.get("cited_by_count"),
                source="openalex",
            )
        )
    return results

In [6]:
def research_search(query: str, max_results: int = 5) -> list[Paper]:
    papers = (
        search_arxiv(query, max_results)
        + search_openalex(query, max_results)
        + search_semantic_scholar(query, max_results)
    )
    seen, deduped = set(), []
    for p in papers:
        key = p.title.strip().lower()
        if key not in seen:
            seen.add(key)
            deduped.append(p)
    return deduped

In [7]:
from langchain_core.tools import tool


@tool
def research_papers(query: str, max_results: int = 5) -> list[dict]:
    """Search arXiv, OpenAlex and Semantic Scholar for academic papers relevant to a learning topic.
    Use this when the user wants to learn about or research a concept and source
    material is needed to ground an explanation."""
    return [p.model_dump() for p in research_search(query, max_results)]

In [8]:
for p in research_search("retrieval augmented generation", max_results=3):
    print(f"[{p.source}] {p.title} ({p.year}) — {len(p.abstract or '')} char abstract")

⚠️ arXiv search failed (HTTPError); continuing with OpenAlex only
[openalex] Retrieval-Augmented Generation for Large Language Models: A Survey (2023) — 1319 char abstract
[openalex] Active Retrieval Augmented Generation (2023) — 211 char abstract
[openalex] RAGAs: Automated Evaluation of Retrieval Augmented Generation (2024) — 201 char abstract
[semantic_scholar] Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (2020) — 1630 char abstract
